# GAGA — Group Aggregation enhanced Transformer for Graph Fraud Detection

### Neural Networks — Project Report (Colab notebook)

Alessandro Nese — *1990771*  



## 1. Project Aim & Selected Paper

### Selected paper

**Label Information Enhanced Fraud Detection against Low Homophily in Graphs**  (Wang et al.)


### What the paper proposes

Graph-based fraud detection is a **node-classification** problem on
multi-relational graphs. Most GNN fraud detectors silently assume **homophily**
and break down under **low homophily**, which is
exactly the regime fraud lives in: fraudsters *camouflage* by connecting to many
benign nodes, so naive neighbourhood averaging buries the fraud signal in the
benign majority. The paper also notes that **label information** is usually underused.

**GAGA** addresses both with two core ideas:

1. **Group aggregation** — instead of pooling a node's neighbours into one vector,
   split them by label (*benign / fraud / unknown*) and pool each group separately,
   so opposite-class signals stop cancelling out.
2. **Learnable group/hop/relation encodings + a Transformer encoder** — the group
   index *is* the class label, so adding a learnable group encoding injects label
   information directly into the feature space; a vanilla Transformer then
   re-weights the resulting short sequence.


### Aim of this project

- Re-implement GAGA from scratch 
- Reproduce the paper's headline behaviour on the two public datasets (Amazon, YelpChi).
- Analyze the effectiveness of each component of GAGA

## 2. Theoretical Background & Key Concepts

### 2.1 Low homophily — why standard GNNs fail

Homophily is measured by the **edge homophily ratio** $h$, the fraction of edges
whose endpoints share a class:

$$ h = \frac{\big|\{(u,v)\in E : y_u = y_v\}\big|}{|E|} $$

High $h\to 1$ is what message-passing GNNs are built for:
they **smooth** each node toward its neighbours. Under **low homophily** ($h\to 0$)
this smoothing mixes in other-class features and *dilutes* the signal 


### 2.2 Problem setup

A graph $\mathcal{G}(V,E,X,Y)$ with $N$ nodes, $R$ relations (one adjacency per
relation), $d$-dimensional features, and **only a small set of labelled nodes**
(semi-supervised). Labels are binary: $y_v=1$ fraud, $y_v=0$ benign ($C=2$).

GAGA targets two weaknesses of prior work:
- **Challenge 1 — indiscriminate aggregation:** message passing mixes all neighbours
  regardless of label
- **Challenge 2 — labels underused:** labels are only supervision, never input

### 2.3 The GAGA pipeline (five stages)

**1-2. Group aggregation (weight-free preprocessing, done once).**
Start from a standard GNN aggregator. Then **partition the sum by label**: with $C=2$ the groups are
$V^-$ (benign), $V^+$ (fraud) and $V^*$ (masked/unknown). Each group is pooled
separately:

$$ h_i = \frac{1}{\phi(\cdot)} \sum_{u \in V_i} x_u, \qquad \phi(\cdot)=|V_i|^{\alpha} $$

The target node is masked into $V^*$ to prevent **label leakage**. Running this per
hop $k=1\dots K$ and per relation, then prepending the node's own raw feature $h_v$,
produces a flat input sequence $H_s$ of length

$$ S = R \times (P\times K + 1), \qquad P = C+1 = 3. $$

For Amazon/YelpChi ($R=3, K=2, P=3$): **$S = 3\times(3\cdot 2+1) = 21$ tokens.**

**3. Linear projection.** Each of the $S$ group vectors is lifted from feature dim
$d$ to the Transformer hidden dim $d_H$: $X_s = \sigma(\psi(H_s))$.

**4. Learnable encodings.** A Transformer sees its input as an unordered set, so
three learnable lookup tables stamp each token with *where it came from* — and are
summed elementwise onto $X_s$:
- **Hop encoding** $X_h$ — which hop ($0=$ target, $1\dots K$).
- **Relation encoding** $X_r$ — which relation block.
- **Group encoding** $X_g$ — which label group; **this is the label-injection trick**,
  since the group index equals the class label.

$$ X_{in} = X_s + X_h + X_r + X_g $$

**5. Transformer encoder + readout + prediction.** A vanilla multi-head
self-attention encoder re-weights the sequence. GAGA then takes only the
**first (center) token of each relation block**, concatenates them across relations into $z_v$, and an MLP
head produces the fraud probability. Training uses (binary) cross-entropy over
labelled nodes only, plus L2 weight decay.

## 3. Implementation Details

### 3.1 Datasets

Both graphs ship as MATLAB `.mat` files:

| Dataset | Nodes | Feat. dim | Relations | # Relations | Fraud % |
|---------|-------|-----------|-----------|---------|-------|
| **Amazon** | 11,944 | 25 | 3 (`U-P-U`, `U-S-U`, `U-V-U`) | 4,778,824 | ~6.9% |
| **YelpChi** | 45,954 | 32 | 3 (`R-S-R`, `R-T-R`, `R-U-R`) | 4,025,674 | ~14.5% |

Both graphs are **low-homophily** which is the setting GAGA is designed for.

### 3.2 From graph to sequence 

For every node, `gaga/sequence.py` walks the $K$-hop neighbourhood per relation and
summarizes each hop into three group means using **only training labels**:

This is the one expensive step executed only once, cached to `seq_cache/` and reused.

### 3.3 Model (`gaga/model.py`)

- `SequenceEncoder`: linear feature projection (+ReLU) followed by three additive
  `nn.Embedding` tables (hop / relation / group), indexed by fixed per-token patterns
  registered as buffers.
- `nn.TransformerEncoder` of `n_layers` standard layers (`batch_first`).
- Readout: take the center token of every relation (stride `1 + K*(C+1)`),
  **concatenate** across relations, classify with a linear head.
- Xavier init; the Amazon model is ~tens of thousands of parameters (printed at train time).


### 3.4 Experimental setup

Optimiser **Adam**, early stopping on **validation AUC** (patience 30). Key hyper-parameters per dataset:

| Config | emb_dim | layers | heads | ff_dim | dropout | lr | batch | hops |
|--------|---------|--------|-------|--------|---------|------|-------|------|
| `amazon.json` | 16 | 2 | 4 | 64 | 0.0 | 1e-3 | 256 | 2 |
| `yelp.json`   | 32 | 3 | 4 | 128 | 0.1 | 1e-3 | 512 | 2 |

Shared: `weight_decay=1e-4`, `max_epochs=500`, `early_stop=30`, `seed=42`,
`train_size=0.4`, `val_size=0.1`, features row-normalized.

## 4. Colab Setup

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q scipy scikit-learn matplotlib tqdm networkx igraph

In [ ]:
import sys, os

REPO_URL = 'https://github.com/Nese2002/GAGA.git'
PROJECT_DIR = '/content/GAGA'

if not os.path.isdir(PROJECT_DIR):
    !git clone -q {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull -q

sys.path.append(PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())
print('Contents   :', sorted(os.listdir(PROJECT_DIR)))

## 5. Exploratory Visualization of the Graphs

We visualize a **subset of the graph** of each dataset,
coloured **red = fraud / black = benign**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import deque
from gaga.data import load_dataset

REL_NAMES = {'amazon': ['U-P-U', 'U-S-U', 'U-V-U'],
             'yelp':   ['R-S-R', 'R-T-R', 'R-U-R']}

def subset_graph(name, n_nodes=300, n_seeds=5, seed=0):
    """Load a dataset and grow a small subgraph (union of all relations) from a
    few fraud seeds, so the fraud-among-benign camouflage is visible."""
    d = load_dataset(name, norm_feat=False)
    y = d['labels']

    A = sum(d['adjacencies']).tocsr()         
    A = ((A + A.T) > 0).astype(np.int8)         
    A.setdiag(0); A.eliminate_zeros()

    rng = np.random.RandomState(seed)
    seeds = rng.choice(np.where(y == 1)[0], n_seeds, replace=False)
    chosen, frontier = set(seeds.tolist()), deque(seeds.tolist())
    while frontier and len(chosen) < n_nodes:
        for u in A[frontier.popleft()].indices:
            if len(chosen) >= n_nodes:
                break
            if u not in chosen:
                chosen.add(u); frontier.append(u)

    nodes = np.array(sorted(chosen))
    return nodes, y[nodes], A[nodes][:, nodes]

def plot_subset(name, ax, **kw):
    nodes, yy, sub = subset_graph(name, **kw)
    G = nx.from_scipy_sparse_array(sub)
    colors = ['red' if c == 1 else 'black' for c in yy]
    pos = nx.spring_layout(G, seed=0)
    nx.draw(G, pos, ax=ax, node_color=colors, node_size=20, edge_color='lightgray',
            width=0.4, with_labels=False)
    ax.set_title(f'{name}  —  {len(nodes)} nodes, {sub.nnz // 2} edges, '
                 f'{int((yy == 1).sum())} fraud')
    ax.axis('off')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_subset('amazon', axes[0])
plot_subset('yelp',   axes[1])
fig.suptitle('Subset of each review graph (union of all relations) — '
             'red = fraud, black = benign', fontsize=14)
plt.tight_layout(); plt.show()

## 6. Training

### 6.1 Amazon


In [ ]:
import json
from gaga.trainer import train, train_multiple

with open('configs/amazon.json') as f:
    config = json.load(f)
config

test_metrics, history = train(config)

### 6.2 YelpChi


In [ ]:
with open('configs/yelp.json') as f:
    yelp_config = json.load(f)

yelp_metrics, yelp_history = train(yelp_config)

### 6.3 Load a saved model & evaluate

In [ ]:
import json
from gaga.trainer import evaluate

# Pick which dataset to evaluate: 'amazon' or 'yelp'
EVAL_DATASET = 'amazon'

with open(f'configs/{EVAL_DATASET}.json') as f:
    eval_config = json.load(f)

eval_metrics = evaluate(eval_config)


## 7. Results & Analysis

### 7.1 Training curves


In [ ]:
from gaga.plot import plot_history
_ = plot_history(history, save_path='checkpoints/amazon_curves.png', show=True)

In [ ]:
_ = plot_history(yelp_history, save_path='checkpoints/yelp_curves.png', show=True)

### 7.2 Metrics table

I report the headline GAGA metrics on the held-out **test** set: **AUC**,
**F1-macro**, **G-Mean**, **AP** (average precision), and the fraud-class
precision/recall. The cell below collects them from both runs into one table.

In [ ]:
import pandas as pd

rows = {'Amazon': test_metrics, 'YelpChi': yelp_metrics}
cols = ['auc', 'f1_macro', 'gmean', 'ap', 'precision_1', 'recall_1', 'f1_fraud']
df = pd.DataFrame(rows).T[cols].round(4)
df.columns = ['AUC', 'F1-macro', 'GMean', 'AP', 'Precision(fraud)', 'Recall(fraud)', 'F1(fraud)']
df


### 7.4 Multiple runs (mean ± std)


In [ ]:
_ = train_multiple(config, n_runs=5)

### 7.5 Interpretation

- **AUC vs F1.** Both datasets are **imbalanced** (Amazon ~6.9% fraud, YelpChi
  ~14.5%), so AUC and F1-macro matter more than raw accuracy
- **Why group aggregation helps.** a fraud node's neighbours are mostly
  benign; splitting neighbours by label keeps the fraud and benign signals in separate
  tokens instead of averaging them away. The group encoding then lets attention treat
  "this is a fraud-group token" as a first-class feature.
- **Cost profile.** Aggregation is weight-free and cached; the trainable model is a
  small Transformer over only 21 tokens, so training is fast and memory-light
  relative to deep message-passing GNNs

## 8. Ablation Study — Effect of Each GAGA Component 

To understand *why* GAGA works, I tried to
switch individual components on and off and re-training. Each variant is trained
**once with the same fixed seed**.


### 8.1 Setup — controls and helpers



In [ ]:
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gaga.trainer import train

ABLATION_DATASET = 'amazon'   
ABL_MAX_EPOCHS   = 200        

with open(f'configs/{ABLATION_DATASET}.json') as f:
    BASE_CFG = json.load(f)
BASE_CFG['max_epochs'] = ABL_MAX_EPOCHS
print(f"Ablation dataset={ABLATION_DATASET} | seed={BASE_CFG['seed']} | max_epochs={ABL_MAX_EPOCHS}")

def _slug(s):
    return re.sub(r'[^0-9a-zA-Z]+', '_', s).strip('_').lower()

def run_variants(variants, base=None):
    """Train each variant once, with the same fixed seed, and collect test metrics.

    variants : list of (label, overrides) where overrides patches the base config.
    Returns a DataFrame indexed by label with AUC / AP / F1-macro columns.
    """
    base = BASE_CFG if base is None else base
    rows = []
    for label, ov in variants:
        cfg = {**base, **ov, 'run_name': f"abl_{ABLATION_DATASET}_{_slug(label)}"}
        print(f"\n{'='*14} {label} {'='*14}")
        m, _ = train(cfg)
        rows.append({'variant': label, 'AUC': m['auc'],
                     'AP': m['ap'], 'F1_macro': m['f1_macro']})
    return pd.DataFrame(rows).set_index('variant').round(4)

def plot_metric(df, metric='AUC', title=''):
    ax = df[metric].plot.bar(figsize=(max(7, 1.4 * len(df)), 4), color='#4c72b0')
    ax.set_ylabel(f'test {metric}'); ax.set_title(title or metric)
    ax.set_ylim(max(0.0, df[metric].min() - 0.05), min(1.0, df[metric].max() + 0.03))
    plt.xticks(rotation=25, ha='right'); plt.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

### 8.2 Full GAGA — all components (reference)

The complete model: group aggregation **on**, all three learnable encodings **on**,
Transformer backbone. Every later ablation is measured against this row.

In [ ]:
full_df = run_variants([('GAGA (all components)', {})])
full_df

### 8.3 Effect of Group Aggregation 

**With GA**, each hop's neighbours are split by their
training label into benign / fraud / unknown group-means, so opposite-class signals
do not cancel. **Without GA** (`group_agg=False`) we fall back to a single mean over all
neighbours 

In [ ]:
ga_df = run_variants([
    ('with GA (GAGA)',          {}),
    ('w/o GA (mean aggregator)', {'group_agg': False}),
])
display(ga_df)
plot_metric(ga_df, 'AUC', f'{ABLATION_DATASET}: effect of group aggregation')

### 8.4 Effect of the Backbone Encoder 

Group aggregation can be paired with different encoders.
Here we keep GA and all encodings on and swap only the sequence model:

- **Transformer** — GAGA's multi-head self-attention re-weights the group tokens (default).
- **MLP** — a token-wise feed-forward network: same capacity per token but **no attention**,
  so tokens cannot interact.
- **none** — skip the encoder entirely; the center tokens go straight to the linear head.

This isolates how much the *self-attention* contributes on top of group aggregation

In [ ]:
backbone_df = run_variants([
    ('Transformer (GAGA)',    {'backbone': 'transformer'}),
    ('MLP (no attention)',    {'backbone': 'mlp'}),
    ('none (linear readout)', {'backbone': 'none'}),
])
display(backbone_df)
plot_metric(backbone_df, 'AUC', f'{ABLATION_DATASET}: effect of the backbone encoder')

### 8.5 Effect of the Learnable Encodings

The three additive encodings stamp each token with *where it came from*: the group
($X_g$), the relation ($X_r$) and the hop ($X_h$). Here I try to demonstrate the effectiveness of the learnable encodings.

In [ ]:
enc_variants = [
    # ---- with group aggregation ----
    ('w/ GA: Xg,Xr,Xh', {}),
    ('w/ GA: Xr,Xh',    {'use_group': False}),
    ('w/ GA: Xg',       {'use_relation': False, 'use_hop': False}),
    ('w/ GA: Xr',       {'use_group': False, 'use_hop': False}),
    ('w/ GA: Xh',       {'use_group': False, 'use_relation': False}),
    ('w/ GA: -',        {'use_group': False, 'use_relation': False, 'use_hop': False}),
    # ---- without group aggregation (Xg not applicable) ----
    ('w/o GA: Xr,Xh',   {'group_agg': False}),
    ('w/o GA: Xr',      {'group_agg': False, 'use_hop': False}),
    ('w/o GA: Xh',      {'group_agg': False, 'use_relation': False}),
    ('w/o GA: -',       {'group_agg': False, 'use_relation': False, 'use_hop': False}),
]
enc_df = run_variants(enc_variants)
display(enc_df[['AUC', 'AP', 'F1_macro']])
plot_metric(enc_df, 'AUC', f'{ABLATION_DATASET}: learnable-encoding ablation (Table 5)')

### 8.7 Takeaways

Putting the ablations together:

- **Group aggregation is the biggest lever**: removing it drops AUC and especially
  **AP** the most: it is what defeats the fraud-node camouflage. 
- **Self-attention adds a clear margin on top of GA**: the Transformer beats the
  no-attention MLP, since attention can re-weight the group tokens per node.
- **The group encoding $X_g$ is the single most valuable encoding**: injecting the
  label as a feature is the core trick, relation/hop encodings help but less.


## 9. Limitations & Reflections

Overall I did not find many serious limitations — GAGA reproduced cleanly and behaved as
the paper describes. The main practical drawback I hit is the **preprocessing/caching cost**:
group aggregation is computed once against a *fixed* label split and cached, so any change to
the split, seed or number of hops forces a full rebuild of the sequences.

**Reflections.** The main takeaway is that *how* labels enter the model can matter more than
model depth: a shallow Transformer over a cleverly grouped 21-token sequence matches deep
message-passing GNNs in the low-homophily regime. The key engineering lesson was avoiding
label leakage in the grouping step by masking the target node into the *unknown* group.


## 10. References


1. Y. Wang et al. *Label Information Enhanced Fraud Detection against Low
   Homophily in Graphs.* WWW 2023. 
2. A. Vaswani et al. *Attention Is All You Need.* NeurIPS 2017.
3. W. Hamilton, R. Ying, J. Leskovec. *Inductive Representation Learning on Large
   Graphs (GraphSAGE).* NeurIPS 2017.
4. Y. Dou et al. *Enhancing Graph Neural Network-based Fraud Detectors against
   Camouflaged Fraudsters (CARE-GNN).* CIKM 2020. — source of the YelpChi/Amazon graphs.
